In [ ]:
# manual_sync.py
import requests

# Lấy URL từ Azure Portal - Dev chatbot

function_url = ""
try:
    # Gửi yêu cầu POST (hoặc GET, vì hàm không yêu cầu phương thức cụ thể)
    response = requests.post(function_url, timeout=10) # timeout ngắn vì ta chỉ cần khởi động

    if response.status_code == 202:
        print("Thành công! Quá trình đồng bộ hóa đã được bắt đầu.")
        print(response.text)
    else:
        print(f"Lỗi khi gọi Function App: {response.status_code}")
        print(response.text)

except requests.exceptions.RequestException as e:
    print(f"Lỗi kết nối: {e}")

Thành công! Quá trình đồng bộ hóa đã được bắt đầu.
Manual change detection process started successfully.


In [ ]:
import json
from urllib.parse import unquote
from azure.storage.blob import BlobServiceClient

# Thông tin kết nối của bạn (giữ nguyên như cũ)
connect_str = ""
container_name = "raw-documents"

# Khởi tạo client
blob_service_client = BlobServiceClient.from_connection_string(connect_str)
container_client = blob_service_client.get_container_client(container_name)

# Hàm giải mã toàn bộ URL-encoded trong metadata
def decode_sharepoint_metadata(metadata):
    decoded = {}
    for key, value in metadata.items():
        if isinstance(value, str):
            # Giải mã %20, %E1%BA%A3... thành ký tự đúng
            decoded[key] = unquote(value)
        else:
            decoded[key] = value
    return decoded

# Lưu metadata đã giải mã
all_decoded_metadata = {}

# Duyệt qua tất cả file và xử lý
for blob in container_client.list_blobs():
    blob_client = container_client.get_blob_client(blob.name)
    properties = blob_client.get_blob_properties()
    
    # Giải mã metadata cho từng file
    decoded_meta = decode_sharepoint_metadata(properties.metadata)
    
    all_decoded_metadata[blob.name] = {
        "file_size_bytes": properties.size,
        "last_modified": str(properties.last_modified),
        "decoded_sharepoint_metadata": decoded_meta
    }

# Xuất ra file JSON mới (không ghi đè file cũ)
output_file = "metadata_30files_vblq_update_32files.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(all_decoded_metadata, f, ensure_ascii=False, indent=4)

print(f"✅ Đã giải mã và xuất metadata ra file: {output_file}")
print(f"📊 Tổng số file đã xử lý: {len(all_decoded_metadata)}")

✅ Đã giải mã và xuất metadata ra file: metadata_30files_vblq_update_32files.json
📊 Tổng số file đã xử lý: 2


In [ ]:
# note:

# func azure functionapp publish <Tên-Function-App-Của-Bạn>

# 1)
# mới thêm 

# document = build_document(metadata)

# postgres_sync.sync(document)

# trong pipeline.py

# 2) 
# mới thêm

# if blob_path: ## if blob_path is not None
#     postgres_sync.delete(blob_path) ## delete document from postgres

# 3) 
# mới thêm
# conn = psycopg.connect(  ## type: add new connection
#     host=settings.host,
#     port=settings.port,
#     dbname=settings.dbname,
#     user=settings.user,
#     password=settings.password,
# )

# 4)
        # if conn: ## close connection if it exists and is not None
        #     conn.close()

# 5)     conn: psycopg2.extensions.connection, ## postgresql connection

# 6) _process_change(item, event_type, run_id, sharepoint, storage, conn, metrics, run_context) ## process change add conn as parameter

# Function App
#     → Development Tools
#         → SSH